## Imports

In [28]:
!pip install -q torchao torchinfo 

In [3]:
import torch
print(torch.__version__)

import torchvision
import torchao  # This should now import from your cloned version
import torch.optim as optim
from torchvision import models, datasets, transforms
import torch.nn as nn
from torchinfo import summary
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from copy import deepcopy
from torchao.quantization.quant_api import (
    quantize_,
    # int8_dynamic_activation_int8_weight,
    # Int4WeightOnlyQuantize,
    int4_weight_only,
    int8_weight_only,
    int8_dynamic_activation_int8_weight,
)


2.5.1+cu124


## Device

In [30]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using {device} device")

Using cuda device


## Loading the dataset

In [31]:
# Load and preprocess CIFAR-100

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

# Set a manual seed
torch.manual_seed(2)

trainset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
testset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

# For now I will keep only 100% of train dataset
# train_size = int(0.6 * len(trainset))
# test_size = len(trainset) - train_size
# trainset, _ = torch.utils.data.random_split(trainset, [train_size, test_size])

print(f"Train set size: {len(trainset)}")
print(f"Test set size: {len(testset)}")

Files already downloaded and verified
Files already downloaded and verified
Train set size: 50000
Test set size: 10000


## Load pretrained VGG-11

In [32]:
# model = models.vgg11(pretrained=True)

from torchvision.models import VGG11_Weights
model = models.vgg11(weights=VGG11_Weights.DEFAULT)

Replace the classification head

In [33]:
# Freeze all the layers in the feature extractor
# for param in model.parameters():
#     param.requires_grad = False

# Modify the classifier to have an output layer with 100 classes
model.classifier[6] = nn.Linear(in_features=4096, out_features=100)

# Move the model to device
model = model.to(device)

# freeze all features parameters
# for param in model.features.parameters():
#     param.requires_grad = False

# Print the model to verify changes
print(model)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [34]:
# print the model summary

summary(model, input_size=(1, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
VGG                                      [1, 100]                  --
├─Sequential: 1-1                        [1, 512, 7, 7]            --
│    └─Conv2d: 2-1                       [1, 64, 224, 224]         1,792
│    └─ReLU: 2-2                         [1, 64, 224, 224]         --
│    └─MaxPool2d: 2-3                    [1, 64, 112, 112]         --
│    └─Conv2d: 2-4                       [1, 128, 112, 112]        73,856
│    └─ReLU: 2-5                         [1, 128, 112, 112]        --
│    └─MaxPool2d: 2-6                    [1, 128, 56, 56]          --
│    └─Conv2d: 2-7                       [1, 256, 56, 56]          295,168
│    └─ReLU: 2-8                         [1, 256, 56, 56]          --
│    └─Conv2d: 2-9                       [1, 256, 56, 56]          590,080
│    └─ReLU: 2-10                        [1, 256, 56, 56]          --
│    └─MaxPool2d: 2-11                   [1, 256, 28, 28]          -

## Finetuning Base Model

In [37]:
# Training function

def train_model(model, train_loader, criterion, optimizer, device, epochs=10, dtype="default"):
    model = model.to(device)
    model.train()
    epoch_losses = []
    all_preds = []
    all_labels = []

    for epoch in range(epochs):
        running_loss = 0.0
        epoch_preds = []
        epoch_labels = []

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch"):
            inputs, labels = inputs.to(device), labels.to(device)
            if dtype == "fp16":
                inputs = inputs.half()
            elif dtype == "bf16":
                inputs = inputs.to(torch.bfloat16)
                
            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            if loss.isnan().any():
                print("Loss is NaN.")

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            # Track loss and predictions
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)

            # Collect predictions and labels for the epoch
            epoch_preds.extend(predicted.cpu().numpy())
            epoch_labels.extend(labels.cpu().numpy())

        # Calculate and store average loss for the epoch
        epoch_loss = running_loss / len(train_loader)
        epoch_losses.append(epoch_loss)

        # Store predictions and labels from the last epoch for further analysis
        all_preds = epoch_preds
        all_labels = epoch_labels

        # Get epoch accuracy
        total_correct = np.sum(np.array(epoch_preds) == np.array(epoch_labels))
        total_samples = len(epoch_labels)
        epoch_accuracy = 100 * total_correct / total_samples

        # Print epoch loss
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")

    return epoch_losses, all_preds, all_labels

# Evaluation function
def evaluate_model(model, test_loader, criterion, device):
    model.eval()
    total_correct = 0
    total_samples = 0
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating", unit="batch"):
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            # Get predictions and accumulate them
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            # Collect predictions and labels
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100 * total_correct / total_samples
    avg_loss = running_loss / len(test_loader)
    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

    return avg_loss, accuracy, all_preds, all_labels

In [38]:
# Define loss function and optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

In [16]:
# Train the model

train_losses, train_preds, train_labels = train_model(model, train_loader, criterion, optimizer, device, epochs=3)
torch.save(model.state_dict(), 'fine_tuned_vgg11.pth')


In [16]:
# Evaluate the model

test_loss, test_accuracy, test_preds, test_labels = evaluate_model(model, test_loader, criterion, device)

Evaluating: 100%|██████████| 157/157 [00:12<00:00, 13.05batch/s]

Test Loss: 1.1297, Test Accuracy: 68.61%


## Loading Already Finetuned Model

In [36]:
# model = models.vgg11(pretrained=False)  # Set to False if loading fine-tuned weights

# # Prompt to upload the file
# from google.colab import files
# uploaded = files.upload()
#  -----------------------------> Faster to upload from the sidebar

# Load the saved weights
# model.load_state_dict(torch.load('/kaggle/input/finetuned-vgg11/pytorch/default/1/fine_tuned_vgg11.pth', map_location=torch.device('cpu')))
model.load_state_dict(torch.load('/kaggle/input/finetuned-vgg11/pytorch/default/6/fine_tuned_vgg11 (4).pth'))
# model.load_state_dict(torch.load('/kaggle/working/fine_tuned_vgg11.pth'))
# model.load_state_dict(torch.load('fine_tuned_vgg11 (4).pth'))

# model.eval()  # Set to evaluation mode if not further training

/tmp/ipykernel_30/1160497649.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('/kaggle/input/finetuned-vgg11/pytorch/default/6/fine_tune

<All keys matched successfully>

Memory Footprint

In [40]:
# # TODO: get the memory footprint of base model

def get_memory_footprint(model):
    memory = 0
    for param in model.parameters():
        memory += param.numel() * param.element_size()
    return memory / (1024) / (1024)

base_mem_footprint = get_memory_footprint(model)
print(f"Memory footprint of base model: {base_mem_footprint :.2f} MB")

Memory footprint of base model: 492.77 MB


# PTQ

## fp16

In [26]:
# cast the model to fp16 dtype. using deepcopy

model_fp16 = deepcopy(model)
model_fp16 = model_fp16.to(torch.float16)

model_fp16 = model_fp16.to(device)

# Define loss function
criterion = nn.CrossEntropyLoss()


In [27]:
# Evaluation function
def evaluate_model_fp_16(model, test_loader, criterion, device):
    model.eval()
    total_correct = 0
    total_samples = 0
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating", unit="batch"):
            inputs, labels = inputs.half().to(device), labels.to(device)
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            # Get predictions and accumulate them
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            # Collect predictions and labels
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100 * total_correct / total_samples
    avg_loss = running_loss / len(test_loader)
    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")
    return avg_loss, accuracy, all_preds, all_labels

In [28]:
fp16_loss, fp16_accuracy, fp16_preds, fp16_labels = evaluate_model_fp_16(model_fp16, test_loader, criterion, device)

Evaluating: 100%|██████████| 157/157 [00:26<00:00,  5.94batch/s]

Test Loss: 1.1297, Test Accuracy: 68.58%


In [29]:
fp16_mem_footprint = get_memory_footprint(model_fp16)

print(f"Memory footprint of fp16 model: {fp16_mem_footprint :.2f} kB")

Memory footprint of fp16 model: 252296.95 kB


## bf16

In [29]:
# cast the model to bf16 using deepcopy

model_bf16 = deepcopy(model)
model_bf16 = model_bf16.to(torch.bfloat16)

model_bf16 = model_bf16.to(device)

# Define the loss function
criterion = nn.CrossEntropyLoss()

In [30]:
# Evaluation function

def evaluate_model_bf_16(model, test_loader, criterion, device):
    model.eval()
    total_correct = 0
    total_samples = 0
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating", unit="batch"):
            inputs, labels = inputs.to(device), labels.to(device)
            inputs = inputs.to(torch.bfloat16)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            # Get predictions and accumulate them
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            # Collect predictions and labels
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100 * total_correct / total_samples
    avg_loss = running_loss / len(test_loader)
    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

    return avg_loss, accuracy, all_preds, all_labels


In [31]:
bf16_loss, bf16_accuracy, bf16_preds, bf16_labels = evaluate_model_bf_16(model_bf16, test_loader, criterion, device)

Evaluating: 100%|██████████| 157/157 [00:10<00:00, 14.54batch/s]

Test Loss: 1.1299, Test Accuracy: 68.52%


In [33]:
bf16_mem_footprint = get_memory_footprint(model_bf16)

print(f"Memory footprint of bf16 model: {bf16_mem_footprint :.2f} kB")

Memory footprint of bf16 model: 252296.95 kB


## int8

In [41]:
# cast the model to int8 using deepcopy
model_int8 = deepcopy(model)
# quantize_(model_int8, int8_weight_only(), device=device)
quantize_(model_int8, int8_dynamic_activation_int8_weight(), device=device)
# model_int8 = torch.compile(model_int8) # max-autotune

model_int8 = model_int8.to(device)

# Define the loss function
criterion = nn.CrossEntropyLoss()

In [42]:
print(model_int8)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [43]:
def evaluate_model_int8(model, test_loader, criterion, device):
    model.eval()
    total_correct = 0
    total_samples = 0
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating", unit="batch"):
            inputs, labels = inputs.to(device), labels.to(device)
            # inputs = inputs.to(torch.int8)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            # Get predictions and accumulate them
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            # Collect predictions and labels
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())


    accuracy = 100 * total_correct / total_samples
    average_loss = running_loss / len(test_loader)
    print(f"Test Loss: {average_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

    return accuracy, average_loss, all_preds, all_labels


In [44]:
int8_accuracy, int8_loss, int8_preds, int8_labels = evaluate_model_int8(model_int8, test_loader, criterion, device)

Evaluating: 100%|██████████| 157/157 [00:39<00:00,  3.93batch/s]

Test Loss: 1.1297, Test Accuracy: 68.53%


In [45]:
def int8_memory_footprint(model):
    total_memory = 0  # in bytes
    for name, param in model.named_parameters():
        # If the parameter is quantized to int8, each element is 1 byte -> if param is from classifier.0 or .3 or .5 layer
        if "classifier" in name:
            memory = param.numel()  # total number of elements in int8 (each takes 1 byte)
        else:
            memory = param.numel() * param.element_size()
        total_memory += memory

    total_memory_kb = total_memory / 1024  # Convert to KB
    total_memory_mb = total_memory_kb / 1024  # Convert to MB

    print(f"\nTotal Memory Footprint: {total_memory_kb:.2f} KB ({total_memory_mb:.2f} MB)")
    return total_memory_kb, total_memory_mb

int8_mem_footprint,_ = int8_memory_footprint(model_int8)
print(f"Memory footprint of int8 model: {int8_mem_footprint :.2f} kB")


Total Memory Footprint: 153161.60 KB (149.57 MB)
Memory footprint of int8 model: 153161.60 kB


## int4

In [17]:
from torchao.quantization.prototype.qat import Int4WeightOnlyQATQuantizer

# Cast model to bfloat16
model_int4 = deepcopy(model).to(torch.bfloat16)

# Apply INT4 quantization 
quantize_(model_int4, int4_weight_only())

# Move quantized model to the device
model_int4 = model_int4.to(device)

In [18]:
print(model_int4)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [19]:
def evaluate_model_int4(model, test_loader, criterion, device):

    model.eval()
    total_correct = 0
    total_samples = 0
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating", unit="batch"):
            inputs, labels = inputs.to(device), labels.to(device)
            inputs = inputs.to(torch.bfloat16)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            # Get predictions and accumulate them
            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            # Collect predictions and labels
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100 * total_correct / total_samples
    average_loss = running_loss / len(test_loader)
    print(f"Test Loss: {average_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

    return accuracy, average_loss, all_preds, all_labels


In [20]:
int4_accuracy, int4_loss, int4_preds, int4_labels = evaluate_model_int4(model_int4, test_loader, criterion, device)

Evaluating: 100%|██████████| 157/157 [00:10<00:00, 14.79batch/s]

Test Loss: 1.1283, Test Accuracy: 68.38%


In [24]:
def int4_memory_footprint(model):
    total_memory = 0  # in bytes
    for name, param in model.named_parameters():
        # If the parameter is quantized to int8, each element is 1 byte -> if param is from classifier.0 or .3 or .5 layer
        if "classifier" in name:
            memory = param.numel()  * 0.5 # total number of elements in int8 (each takes 1 byte)
        else:
            memory = param.numel() * param.element_size()
        total_memory += memory

    total_memory_kb = total_memory / 1024  # Convert to KB
    total_memory_mb = total_memory_kb / 1024  # Convert to MB

    print(f"\nTotal Memory Footprint: {total_memory_kb:.2f} KB ({total_memory_mb:.2f} MB)")
    return total_memory_kb, total_memory_mb


int4_mem_footprint,_ = int4_memory_footprint(model_int4)
print(f"Memory footprint of int4 model: {int4_mem_footprint :.2f} kB")


Total Memory Footprint: 76580.80 KB (74.79 MB)
Memory footprint of int4 model: 76580.80 kB


# QAT

## fp16


In [52]:
optimizer = optim.SGD(model_fp16.parameters(), lr=1e-4)

# Finetune the fp16 model for 3 epochs
finetuned_fp_16_train_losses, finetuned_fp_16_train_preds, finetuned_fp_16_train_labels = train_model(model_fp16, train_loader, criterion, optimizer, device, epochs=3, dtype="fp16")


Epoch 1/3: 100%|██████████| 782/782 [04:22<00:00,  2.98batch/s]


Epoch [1/3], Loss: 0.2181, Accuracy: 94.10%


Epoch 2/3: 100%|██████████| 782/782 [04:20<00:00,  3.00batch/s]


Epoch [2/3], Loss: 0.1983, Accuracy: 94.88%


Epoch 3/3: 100%|██████████| 782/782 [04:20<00:00,  3.00batch/s]

Epoch [3/3], Loss: 0.1901, Accuracy: 95.01%


In [53]:

finetuned_fp16_loss, finetuned_fp16_accuracy, finetuned_fp16_predsszws, finetuned_fp16_labels = evaluate_model_fp_16(model_fp16, test_loader, criterion, device)

Evaluating: 100%|██████████| 157/157 [00:26<00:00,  5.92batch/s]

Test Loss: 1.0645, Test Accuracy: 69.98%


In [54]:
print("Model memory footprint:", get_memory_footprint(model_fp16))

Model memory footprint: 252296.9453125


## bf16

In [32]:
optimizer = optim.SGD(model_bf16.parameters(), lr=1e-4)

# Finetune the fp16 model for 3 epochs
finetuned_bf16_train_losses, finetuned_bf16_train_preds, finetuned_bf16_train_labels = train_model(model_bf16, train_loader, criterion, optimizer, device, epochs=3, dtype="bf16")


Epoch 1/3: 100%|██████████| 782/782 [01:13<00:00, 10.69batch/s]


Epoch [1/3], Loss: 0.2333, Accuracy: 93.71%


Epoch 2/3: 100%|██████████| 782/782 [01:12<00:00, 10.86batch/s]


Epoch [2/3], Loss: 0.2265, Accuracy: 93.96%


Epoch 3/3: 100%|██████████| 782/782 [01:12<00:00, 10.80batch/s]

Epoch [3/3], Loss: 0.2198, Accuracy: 94.24%


In [33]:
finetuned_bf16_loss, finetuned_bf16_accuracy, finetuned_bf16_preds, finetuned_bf16_labels = evaluate_model_bf_16(model_bf16, test_loader, criterion, device)

Evaluating: 100%|██████████| 157/157 [00:10<00:00, 14.95batch/s]

Test Loss: 1.1055, Test Accuracy: 69.02%


In [34]:
print("Model memory footprint:", get_memory_footprint(model_bf16))

Model memory footprint: 252296.9453125


## int8

In [46]:
!pip install -q torchtune

# from torchao.quantization.qat import Int8DynActInt4WeightQATQuantizer

import torchtune
from torchtune.training.quantization import Int8DynActInt4WeightQATQuantizer
from torchao.prototype.quantized_training import int8_weight_only_quantized_training


In [47]:
model_int8_qat = deepcopy(model).to(device)

# Quantizer for int8 dynamic per token activations +
# int4 grouped per channel weights, only for linear layers
quantizer = Int8DynActInt4WeightQATQuantizer()

# Insert "fake quantize" operations into linear layers.
# These operations simulate quantization numerics during
# fine-tuning without performing any dtype casting
prepared_model = quantizer.prepare(model_int8_qat)

# optimizer = _AdamW(prepared_model.parameters(), lr=1e-4)
optimizer = optim.SGD(prepared_model.parameters(), lr=1e-4)

finetuned_int8_train_losses, finetuned_int8_train_preds, finetuned_int8_train_labels = train_model(prepared_model, train_loader, criterion, optimizer, device, epochs=3, dtype="default")

Epoch 1/3: 100%|██████████| 782/782 [07:29<00:00,  1.74batch/s]


Epoch [1/3], Loss: 0.2044, Accuracy: 94.40%


Epoch 2/3: 100%|██████████| 782/782 [07:29<00:00,  1.74batch/s]


Epoch [2/3], Loss: 0.1741, Accuracy: 95.18%


Epoch 3/3: 100%|██████████| 782/782 [07:29<00:00,  1.74batch/s]


Epoch [3/3], Loss: 0.1641, Accuracy: 95.35%


In [48]:
model_int8_qat = quantizer.convert(prepared_model)

finetuned_int8_loss, finetuned_int8_accuracy, finetuned_int8_preds, finetuned_int8_labels = evaluate_model_int8(model_int8_qat, test_loader, criterion, device)

print(model_int8_qat)

Evaluating: 100%|██████████| 157/157 [00:41<00:00,  3.80batch/s]

Test Loss: 1.0844, Test Accuracy: 70.53%
VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, 

## int4

In [ ]:
!pip install -q torchtune
# from torchao.quantization.qat import Int8DynActInt4WeightQATQuantizer
import torchtune
# from torchtune.training.quantization import Int4WeightOnlyQATQuantizer

from torchao.quantization.prototype.qat import Int4WeightOnlyQATQuantizer

In [27]:
# Cast model to bfloat16

model_int4_qat = deepcopy(model).to(torch.bfloat16)
quantizer = Int4WeightOnlyQATQuantizer()

# Insert "fake quantize" operations into linear layers.
# These operations simulate quantization numerics during
# fine-tuning without performing any dtype casting

prepared_model = quantizer.prepare(model_int4_qat)

optimizer = optim.SGD(prepared_model.parameters(), lr=1e-4)

finetuned_int4_train_losses, finetuned_int4_train_preds, finetuned_int4_train_labels = train_model(prepared_model, train_loader, criterion, optimizer, device, epochs=3, dtype="bf16")


Epoch 1/3: 100%|██████████| 782/782 [01:19<00:00,  9.78batch/s]


Epoch [1/3], Loss: 0.2334, Accuracy: 93.61%


Epoch 2/3: 100%|██████████| 782/782 [01:12<00:00, 10.85batch/s]


Epoch [2/3], Loss: 0.2246, Accuracy: 93.97%


Epoch 3/3: 100%|██████████| 782/782 [01:12<00:00, 10.80batch/s]

Epoch [3/3], Loss: 0.2225, Accuracy: 93.96%


In [28]:
model_int4_qat = quantizer.convert(prepared_model)

finetuned_int4_loss, finetuned_int4_accuracy, finetuned_int4_preds, finetuned_int4_labels = evaluate_model_int4(model_int4_qat, test_loader, criterion, device)

print(model_int4_qat)

Evaluating: 100%|██████████| 157/157 [00:11<00:00, 14.27batch/s]

Test Loss: 1.1053, Test Accuracy: 69.02%
VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, 